# CorefInst — Hindi · Tamil · Bengali
**Instruction-tuned LLM for Multilingual Coreference Resolution**

Reproduces *CorefInst: Leveraging LLMs for Multilingual Coreference Resolution* (TACL 2026)  
applied to three Indic languages: **Hindi (hi)**, **Tamil (ta)**, **Bengali (bn)**  
using the **TransMuCoRes** dataset.

---
### Runtime guide
| GPU | VRAM | Recommended config |
|-----|------|--------------------|
| Colab T4 (free) | 16 GB | `configs/t4.yaml` — Llama 3.1 8B 4-bit, batch=2 |
| Colab A100 (Pro) | 40–80 GB | `configs/a100.yaml` — Llama 3.1 8B 4-bit, batch=8 |
| Laptop (CPU/MPS) | — | `configs/cpu.yaml` — Qwen2.5 1.5B, batch=1 |

**Full training time** on T4: ~6–8 h · on A100: ~1.5–2 h

## Step 1 — Check GPU & environment

In [ ]:
import subprocess, sys

# Show GPU info
!nvidia-smi

import torch
print(f"\nPyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"GPU: {props.name}  VRAM: {vram_gb:.1f} GB")
    if vram_gb >= 35:
        PRESET = 'a100'
    else:
        PRESET = 't4'
else:
    PRESET = 'cpu'

print(f"\nSelected hardware preset: {PRESET}")

## Step 2 — Mount Google Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ----------------------------------------------------------------
# EDIT these paths to match your Drive layout
# ----------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/CorefInst'   # project root on Drive
DATA_TAR     = f'{DRIVE_ROOT}/transmucores_data.tar.gz'
PROJECT_DIR  = '/content/corefinst'                 # local working dir
# ----------------------------------------------------------------

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Drive root:  {DRIVE_ROOT}')
print(f'Project dir: {PROJECT_DIR}')

## Step 3 — Install dependencies

In [ ]:
# Unsloth — fast quantization-aware LoRA fine-tuning (designed for Colab)
!pip install -q "unsloth[colab-new]" 2>/dev/null || pip install -q unsloth

# Core ML stack
!pip install -q \
    transformers>=4.44.0 \
    datasets>=2.20.0 \
    peft>=0.12.0 \
    trl>=0.10.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.33.0 \
    scipy \
    PyYAML

print('Installation complete.')

## Step 4 — Clone / upload project code

In [ ]:
import shutil, os

# Option A: copy from Drive (if you uploaded the zip to Drive)
CODE_ZIP = f'{DRIVE_ROOT}/corefinst_code.zip'
if os.path.exists(CODE_ZIP):
    shutil.unpack_archive(CODE_ZIP, PROJECT_DIR)
    print('Code extracted from Drive.')
else:
    # Option B: upload files directly via Colab file upload
    print('ZIP not found on Drive.')
    print('Either: upload corefinst_code.zip to', DRIVE_ROOT)
    print('Or:     use the Colab file panel to upload src/ and scripts manually.')

os.chdir(PROJECT_DIR)
!pip install -q -e . --quiet   # installs src/ as editable package
print('Working directory:', os.getcwd())

## Step 5 — Extract dataset

In [ ]:
import os

DATA_DIR = os.path.join(PROJECT_DIR, 'transmucores_data')

if not os.path.isdir(DATA_DIR):
    if os.path.exists(DATA_TAR):
        print(f'Extracting dataset from {DATA_TAR} …')
        import tarfile
        with tarfile.open(DATA_TAR) as tf:
            tf.extractall(PROJECT_DIR)
        # Extract sub-archives
        for sub in ['mujadia_conll', 'onto_notes', 'litbank_train',
                    'litbank_val', 'litbank_test']:
            gz = os.path.join(DATA_DIR, f'{sub}.tar.gz')
            if os.path.exists(gz):
                with tarfile.open(gz) as tf:
                    tf.extractall(DATA_DIR)
        print('Done.')
    else:
        print(f'Dataset tar not found at {DATA_TAR}')
        print('Upload transmucores_data.tar.gz to', DRIVE_ROOT)
else:
    print(f'Dataset already extracted at {DATA_DIR}')

# Quick count
for subdir in ['mujadia_conll/train', 'litbank_train', 'onto_notes_archive/train']:
    p = os.path.join(DATA_DIR, subdir)
    if os.path.isdir(p):
        n = sum(1 for f in os.listdir(p) if f.endswith('.conll'))
        print(f'  {subdir}: {n} .conll files')

## Step 6 — Select hardware config & prepare data

In [ ]:
import shutil, os

# Copy the appropriate preset to config.yaml
preset_path = os.path.join(PROJECT_DIR, f'configs/{PRESET}.yaml')
config_path = os.path.join(PROJECT_DIR, 'config.yaml')

if os.path.exists(preset_path):
    shutil.copy(preset_path, config_path)
    print(f'Using preset: configs/{PRESET}.yaml')
else:
    print(f'Preset not found at {preset_path} — using existing config.yaml')

# Override data.root and training.output_dir to use Drive for persistence
import yaml
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg['data']['root']              = DATA_DIR
cfg['data']['output_dir']        = os.path.join(DRIVE_ROOT, 'processed_data')
cfg['training']['output_dir']    = os.path.join(DRIVE_ROOT, 'model_output')
cfg['inference']['output_dir']   = os.path.join(DRIVE_ROOT, 'inference_output')

with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

print('Config updated:')
print(f'  data.root       → {cfg["data"]["root"]}')
print(f'  data.output_dir → {cfg["data"]["output_dir"]}')
print(f'  training dir    → {cfg["training"]["output_dir"]}')

In [ ]:
# Preprocess all three languages into JSONL frame examples
# Skips if output already exists on Drive (safe to re-run after session reset)

import os
processed_dir = cfg['data']['output_dir']
train_jsonl   = os.path.join(processed_dir, 'train.jsonl')

if os.path.exists(train_jsonl):
    print(f'Processed data already exists at {processed_dir}')
    print('Delete those files and re-run this cell to regenerate.')
else:
    !python prepare_data.py --config config.yaml

## Step 7 — Train the model

In [ ]:
# ── Training notes ──────────────────────────────────────────────────────────
# • Checkpoints are saved to DRIVE_ROOT/model_output/ every 200 steps.
# • If your Colab session resets, re-run all earlier cells then run this cell
#   again — training will resume from the latest checkpoint automatically.
# • For a quick smoke-test (few-shot mode), change FEW_SHOT = True below.
# ────────────────────────────────────────────────────────────────────────────

FEW_SHOT = False   # True → 5 examples per language (~10 min on T4); False → full training

if FEW_SHOT:
    !python train_model.py --config config.yaml --few_shot 5
else:
    !python train_model.py --config config.yaml

## Step 8 — Run inference & evaluate

In [ ]:
# Evaluate on the test split for all three languages
!python run_inference.py \
    --config config.yaml \
    --split test \
    --language all

## Step 9 — Analysis & ablation

In [ ]:
# Per-language breakdown and ablation over instruction sets
!python analysis/analyse_results.py \
    --results_json inference_output/results.json \
    --config config.yaml

In [ ]:
# Majority-cluster baseline comparison
!python analysis/baseline.py --config config.yaml

## Step 10 — Save everything to Drive
Checkpoints are saved automatically. Run this cell to also copy the inference
outputs and processed data back to Drive.

In [ ]:
import shutil, os

for local_subdir, drive_name in [
    ('inference_output', 'inference_output'),
    ('processed_data',   'processed_data'),
]:
    src = os.path.join(PROJECT_DIR, local_subdir)
    dst = os.path.join(DRIVE_ROOT, drive_name)
    if os.path.isdir(src):
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Saved {local_subdir} → {dst}')

print('All outputs saved to Drive.')